# Classificazione ERP Binaria: GroupKFold Cross Validation & Ottimizzazione Massiva

Questo notebook implementa una pipeline di classificazione robusta per segnali EEG/ERP con l'obiettivo di determinare se il cervello umano distingue tra volti reali e generati dall'IA.

### Architettura della Validazione
A differenza di un singolo split Train/Test (che su 100 campioni produce risultati estremamente instabili), utilizziamo una **GroupKFold Cross Validation** dove ogni fold rispetta il raggruppamento per soggetto, eliminando il Data Leakage.

Ogni modello viene valutato come **media ± deviazione standard** su tutti i fold, producendo una stima affidabile delle performance reali.

### Le 3 Architetture Testate
1. **TemporalGraphNet (GNN)**: Rete a Grafo con matrice di adiacenza apprendibile tra elettrodi.
2. **Res1DCNN**: Rete Convoluzionale 1D con estrazione feature spaziotemporali.
3. **MemoryRNN (LSTM)**: Rete Ricorrente con focus sullo stato finale (componenti tardive P300/LPP).

### Ottimizzazione
Per ciascuna architettura, **Optuna** esegue una ricerca bayesiana degli iperparametri valutando ogni candidato sulla media dei K fold.


In [ ]:
import numpy as np
import os
import random
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                             classification_report, roc_curve, auc)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

# Silenziamo i log intermedi di Optuna per non intasare l'output
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Seed: {SEED}")


## 1. Caricamento Dati e Preprocessing
Mappatura binaria (AI=0, Human=1). Il genere viene conservato per l'analisi post-hoc finale.

In [ ]:
data_path = "../data/file_tensor"
X = np.load(os.path.join(data_path, "x.npy"))
Y_str = np.load(os.path.join(data_path, "y.npy"))
subjects = np.load(os.path.join(data_path, "subjects.npy"))

# Mappatura binaria
label_map = {'50AM': 0, '60AF': 0, '70RM': 1, '80RF': 1}
class_names = ['AI', 'Human']
gender_map_dict = {'50AM': 'Male', '60AF': 'Female', '70RM': 'Male', '80RF': 'Female'}

Y = np.array([label_map[l] for l in Y_str], dtype=np.int64)
Y_gender = np.array([gender_map_dict[l] for l in Y_str])

# Normalizzazione globale per sample
X_mean = X.mean(axis=(1, 2), keepdims=True)
X_std = X.std(axis=(1, 2), keepdims=True)
X_norm = (X - X_mean) / (X_std + 1e-8)

n_subjects = len(np.unique(subjects))
print(f"Dataset: {X.shape[0]} campioni, {X.shape[1]} canali, {X.shape[2]} timesteps")
print(f"Classi: {np.bincount(Y)} (AI / Human)")
print(f"Soggetti unici: {n_subjects}")
print(f"Genere della Faccia (Stimolo): M={np.sum(Y_gender=='Male')}, F={np.sum(Y_gender=='Female')}")

# Definiamo il numero di Fold in base ai soggetti
N_FOLDS = min(n_subjects, 5)
print(f"\nGroupKFold con K={N_FOLDS} fold")


In [ ]:
class EEGDataset(Dataset):
    def __init__(self, x, y, augment=False):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        wave = self.x[idx].clone()
        if self.augment:
            # Gaussian noise
            wave += torch.randn_like(wave) * 0.05
            # Random amplitude scaling ±10%
            wave = wave * (1.0 + (random.random() * 0.2 - 0.1))
        return wave, self.y[idx]


## 2. Definizione delle 3 Architetture
Tutte parametrizzate per consentire la ricerca Optuna sugli iperparametri strutturali.

In [ ]:
class TemporalGraphNet(nn.Module):
    def __init__(self, channels=19, classes=2, temp_filters=32, 
                 kernel_size=8, dropout_rate=0.5):
        super().__init__()
        self.adj = nn.Parameter(torch.ones(channels, channels) / channels)
        self.temp_conv = nn.Conv1d(channels, temp_filters, 
                                   kernel_size=kernel_size, padding='same')
        self.pool = nn.AdaptiveAvgPool1d(8)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(temp_filters * 8, classes)
        
    def forward(self, x):
        x_g = x.transpose(1, 2) @ self.adj
        x_g = x_g.transpose(1, 2)
        x_c = torch.relu(self.temp_conv(x_g))
        x_p = self.pool(x_c)
        x_f = x_p.view(x_p.size(0), -1)
        return self.fc(self.dropout(x_f))


class Res1DCNN(nn.Module):
    def __init__(self, classes=2, filters1=32, filters2=64,
                 kernel_size=5, dropout_rate=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(19, filters1, kernel_size=kernel_size, padding='same')
        self.bn1 = nn.BatchNorm1d(filters1)
        self.pool1 = nn.AvgPool1d(4)
        
        self.conv2 = nn.Conv1d(filters1, filters2, kernel_size=kernel_size, padding='same')
        self.bn2 = nn.BatchNorm1d(filters2)
        self.pool2 = nn.AvgPool1d(4)
        
        self.dropout = nn.Dropout(dropout_rate)
        # Dopo due pool di 4: 205 // 4 = 51, 51 // 4 = 12
        self.pool_final = nn.AdaptiveAvgPool1d(8)
        self.fc = nn.Linear(filters2 * 8, classes)
        
    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool_final(x)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class MemoryRNN(nn.Module):
    def __init__(self, classes=2, hidden_size=64, num_layers=1, 
                 dropout_rate=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size=19, hidden_size=hidden_size, 
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout_rate if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_size, classes)
        
    def forward(self, x):
        x = x.transpose(1, 2)  # [B, C, T] -> [B, T, C]
        out, (hn, cn) = self.lstm(x)
        x = out[:, -1, :]  # Ultimo stato temporale
        return self.fc(self.dropout(x))


## 3. Motore di Addestramento con GroupKFold
Funzione centrale che addestra e valuta un modello su tutti i K fold, restituendo accuracy media e deviazione standard.

In [ ]:
def train_model_kfold(model_factory, epochs=60, batch_size=16):
    """
    Addestra un modello su tutti i K fold e restituisce:
    - mean_acc: media delle accuracy sui fold
    - std_acc: deviazione standard
    - fold_accs: lista delle accuracy per fold
    - best_model_state: pesi del modello del fold migliore
    """
    gkf = GroupKFold(n_splits=N_FOLDS)
    fold_accs = []
    best_fold_acc = 0.0
    best_model_state = None
    
    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_norm, Y, groups=subjects)):
        # Dataset per questo fold
        train_ds = EEGDataset(X_norm[train_idx], Y[train_idx], augment=True)
        test_ds = EEGDataset(X_norm[test_idx], Y[test_idx], augment=False)
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
        
        # Nuovo modello per ogni fold
        model = model_factory().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        best_epoch_acc = 0.0
        best_epoch_weights = None
        
        for epoch in range(epochs):
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(inputs), labels)
                loss.backward()
                optimizer.step()
                
            # Eval
            model.eval()
            correct = 0; total = 0
            with torch.no_grad():
                for inputs, labels in test_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    correct += (model(inputs).argmax(1) == labels).sum().item()
                    total += labels.size(0)
            acc = correct / total
            
            if acc > best_epoch_acc:
                best_epoch_acc = acc
                best_epoch_weights = copy.deepcopy(model.state_dict())
        
        fold_accs.append(best_epoch_acc)
        
        if best_epoch_acc > best_fold_acc:
            best_fold_acc = best_epoch_acc
            best_model_state = best_epoch_weights
    
    return np.mean(fold_accs), np.std(fold_accs), fold_accs, best_model_state


## 4. Arena Benchmark (GroupKFold)
Confronto rapido delle 3 architetture con parametri di default, valutate sulla media dei K fold.

In [ ]:
arena_models = {
    "GNN": lambda: TemporalGraphNet(),
    "CNN 1D": lambda: Res1DCNN(),
    "RNN (LSTM)": lambda: MemoryRNN()
}

arena_results = {}
print("=" * 60)
print("ARENA BENCHMARK (GroupKFold, Default Hyperparameters)")
print("=" * 60)

for name, factory in arena_models.items():
    t0 = time.time()
    mean_acc, std_acc, fold_accs, _ = train_model_kfold(factory, epochs=60)
    elapsed = time.time() - t0
    arena_results[name] = {'mean': mean_acc, 'std': std_acc, 'folds': fold_accs}
    print(f"  {name:15s} -> {mean_acc*100:.1f}% ± {std_acc*100:.1f}%  "
          f"[Folds: {[f'{a*100:.0f}%' for a in fold_accs]}]  ({elapsed:.1f}s)")

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
names = list(arena_results.keys())
means = [arena_results[n]['mean'] for n in names]
stds = [arena_results[n]['std'] for n in names]
colors = ['purple', 'salmon', 'lightgreen']

bars = ax.bar(names, means, yerr=stds, capsize=8, color=colors, edgecolor='black', linewidth=0.8)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.02,
            f'{m*100:.1f}±{s*100:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Test Accuracy (GroupKFold Mean)')
ax.set_title('Benchmark Architetture con Cross-Validation Robusta')
ax.set_ylim([0, 1.1])
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance Level (50%)')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Permutation Importance (sul modello vincitore dell'Arena)
Identifichiamo le finestre temporali critiche per la classificazione.

In [ ]:
# Troviamo il vincitore dell'arena
best_arena_name = max(arena_results, key=lambda k: arena_results[k]['mean'])
print(f"Modello vincitore Arena: {best_arena_name}")

# Riaddestriamo il vincitore su un singolo split per la permutation importance
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
pi_train_idx, pi_test_idx = next(gss.split(X_norm, Y, groups=subjects))

pi_train_ds = EEGDataset(X_norm[pi_train_idx], Y[pi_train_idx], augment=True)
pi_test_ds = EEGDataset(X_norm[pi_test_idx], Y[pi_test_idx], augment=False)
pi_train_loader = DataLoader(pi_train_ds, batch_size=16, shuffle=True)
pi_test_loader = DataLoader(pi_test_ds, batch_size=16, shuffle=False)

# Ricostruiamo il modello vincitore
if best_arena_name == "GNN":
    pi_model = TemporalGraphNet().to(device)
elif best_arena_name == "CNN 1D":
    pi_model = Res1DCNN().to(device)
else:
    pi_model = MemoryRNN().to(device)

optimizer = optim.Adam(pi_model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(60):
    pi_model.train()
    for inputs, labels in pi_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(pi_model(inputs), labels)
        loss.backward()
        optimizer.step()

# Permutation Importance
pi_model.eval()
all_inputs = torch.cat([x for x, _ in pi_test_loader]).to(device)
all_labels = torch.cat([y for _, y in pi_test_loader]).to(device)

with torch.no_grad():
    acc_baseline = accuracy_score(all_labels.cpu(), pi_model(all_inputs).argmax(1).cpu())

window_size = 41
drops, window_labels = [], []
for w in range(5):
    s, e = w * window_size, (w + 1) * window_size
    X_corrupt = all_inputs.clone()
    X_corrupt[:, :, s:e] = torch.randn_like(X_corrupt[:, :, s:e]) * all_inputs.std()
    with torch.no_grad():
        acc_c = accuracy_score(all_labels.cpu(), pi_model(X_corrupt).argmax(1).cpu())
    drops.append(acc_baseline - acc_c)
    ms_start = round(200 + s * (400 / 205))
    ms_end   = round(200 + e * (400 / 205))
    window_labels.append(f"W{w+1}\n({ms_start}-{ms_end} ms)")

plt.figure(figsize=(7, 4))
plt.plot(window_labels, drops, 'ro-', linewidth=2, markersize=8)
plt.fill_between(window_labels, drops, color='red', alpha=0.1)
plt.title(f'Permutation Importance ({best_arena_name})')
plt.ylabel('Drop Accuracy')
plt.xlabel('Finestre Temporali (ms da onset stimolo)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


---
# SEZIONE A: Ottimizzazione TemporalGraphNet (GNN)
Ricerca Bayesiana massiva degli iperparametri con validazione GroupKFold su ogni trial.

In [ ]:
# ============================================================
# SEZIONE A: Nested CV — TemporalGraphNet (GNN)
# Outer: GroupKFold(5) per stima finale
# Inner: GroupShuffleSplit(3) per Optuna (per ogni outer fold)
# ============================================================
from sklearn.metrics import roc_auc_score

N_INNER_TRIALS = 100  
gnn_outer_models = []  

outer_gkf = GroupKFold(n_splits=N_FOLDS)

gnn_outer_accs = []
gnn_fold_aucs = []  
gnn_best_params_per_fold = []
gnn_all_preds, gnn_all_labels, gnn_all_probs, gnn_all_genders = [], [], [], []

print("=" * 60)
print("SEZIONE A: Nested CV GNN")
print(f"  Outer: GroupKFold({N_FOLDS})")
print(f"  Inner: GroupShuffleSplit(3) × Optuna({N_INNER_TRIALS} trials)")
print("=" * 60)

t0 = time.time()

for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_gkf.split(X_norm, Y, groups=subjects)):

    print(f"\n[Outer Fold {outer_fold+1}/{N_FOLDS}] "
          f"Train={len(outer_train_idx)}, Test={len(outer_test_idx)}")

    # ----------------------------------------------------------
    # INNER LOOP: Optuna minimizza la Val Loss media su 3 split
    # ----------------------------------------------------------
    def objective_gnn(trial):
        temp_filters  = trial.suggest_categorical('temp_filters', [16, 32, 64, 128])
        kernel_size   = trial.suggest_categorical('kernel_size', [4, 8, 16, 32])
        dropout_rate  = trial.suggest_float('dropout_rate', 0.1, 0.8)
        lr            = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
        weight_decay  = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        epochs        = trial.suggest_categorical('epochs', [60, 80, 100])

        # Aumentato a 3 split per maggiore stabilità statistica
        gss_inner = GroupShuffleSplit(n_splits=3, test_size=0.25,
                                      random_state=SEED + outer_fold)
        
        split_val_losses = []

        for local_tr, local_val in gss_inner.split(
                X_norm[outer_train_idx], Y[outer_train_idx], groups=subjects[outer_train_idx]):
            
            inner_train_idx = outer_train_idx[local_tr]
            inner_val_idx   = outer_train_idx[local_val]

            train_ds = EEGDataset(X_norm[inner_train_idx], Y[inner_train_idx], augment=True)
            val_ds   = EEGDataset(X_norm[inner_val_idx],   Y[inner_val_idx],   augment=False)
            train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
            val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

            model = TemporalGraphNet(
                temp_filters=temp_filters,
                kernel_size=kernel_size,
                dropout_rate=dropout_rate
            ).to(device)

            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = nn.CrossEntropyLoss()

            for epoch in range(epochs):
                model.train()
                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()
                    criterion(model(inputs), labels).backward()
                    optimizer.step()

            # Valutazione della Loss alla fine delle epoche previste
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * inputs.size(0)
            
            val_loss /= len(val_loader.dataset)
            split_val_losses.append(val_loss)

        # Ritorna la media della validazione sui 3 split interni
        return np.mean(split_val_losses)

    # DIRECTION = MINIMIZE perché ora ottimizziamo la Loss
    study = optuna.create_study(direction='minimize',
                                study_name=f'GNN_outer{outer_fold}')
    study.optimize(objective_gnn, n_trials=N_INNER_TRIALS, n_jobs=1)

    best_p = study.best_trial.params
    gnn_best_params_per_fold.append(best_p)
    print(f"  Inner best val loss: {study.best_trial.value:.4f}  |  HP: {best_p}")

    # ----------------------------------------------------------
    # OUTER EVAL: riaddestra su tutto outer_train ciecamente
    # ----------------------------------------------------------
    torch.manual_seed(SEED + outer_fold)

    model_final = TemporalGraphNet(
        temp_filters=best_p['temp_filters'],
        kernel_size=best_p['kernel_size'],
        dropout_rate=best_p['dropout_rate']
    ).to(device)

    outer_train_ds = EEGDataset(X_norm[outer_train_idx], Y[outer_train_idx], augment=True)
    outer_test_ds  = EEGDataset(X_norm[outer_test_idx],  Y[outer_test_idx],  augment=False)
    outer_train_loader = DataLoader(outer_train_ds, batch_size=16, shuffle=True)
    outer_test_loader  = DataLoader(outer_test_ds,  batch_size=16, shuffle=False)

    optimizer = optim.Adam(model_final.parameters(),
                           lr=best_p['lr'], weight_decay=best_p['weight_decay'])
    criterion = nn.CrossEntropyLoss()

    for epoch in range(best_p['epochs']):
        model_final.train()
        for inputs, labels in outer_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            criterion(model_final(inputs), labels).backward()
            optimizer.step()

    # VALUTAZIONE FINALE: Il test set viene visto solo qui
    model_final.eval()
    c = 0; t = 0
    fold_probs, fold_labels, fold_preds = [], [], []
    
    with torch.no_grad():
        for inputs, labels in outer_test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            out = model_final(inputs)
            preds = out.argmax(1)
            
            c += (preds == labels).sum().item()
            t += labels.size(0)
            
            fold_probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            fold_preds.extend(preds.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())
            
    best_outer_acc = c / t
    
    try:
        fold_auc = roc_auc_score(fold_labels, fold_probs)
    except ValueError:
        fold_auc = 0.5 
        
    gnn_outer_accs.append(best_outer_acc)
    gnn_fold_aucs.append(fold_auc)
    gnn_outer_models.append(copy.deepcopy(model_final))
    
    gnn_all_probs.extend(fold_probs)
    gnn_all_preds.extend(fold_preds)
    gnn_all_labels.extend(fold_labels)
    gnn_all_genders.extend(Y_gender[outer_test_idx])
    
    print(f"  Outer test acc: {best_outer_acc*100:.1f}% (AUC: {fold_auc:.3f})")

In [ ]:
gnn_all_preds   = np.array(gnn_all_preds)
gnn_all_labels  = np.array(gnn_all_labels)
gnn_all_probs   = np.array(gnn_all_probs)
gnn_all_genders = np.array(gnn_all_genders)

elapsed_gnn = time.time() - t0
print(f"\n{'='*60}")
print(f"GNN Nested CV Accuracy: "
      f"{np.mean(gnn_outer_accs)*100:.1f}% ± {np.std(gnn_outer_accs)*100:.1f}%")
print(f"Fold Details: {[f'{a*100:.0f}%' for a in gnn_outer_accs]}")
print(f"Tempo totale: {elapsed_gnn/60:.1f} min")
print()
print(classification_report(gnn_all_labels, gnn_all_preds, target_names=class_names))

# Visualizzazioni (identiche a prima — nessuna modifica)
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
cm = confusion_matrix(gnn_all_labels, gnn_all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('GNN: Confusion Matrix (Nested CV)')

mask_m = gnn_all_genders == 'Male'
mask_f = gnn_all_genders == 'Female'
cm_m = confusion_matrix(gnn_all_labels[mask_m], gnn_all_preds[mask_m], labels=[0,1])
sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title(f'GNN: Facce Maschili ({mask_m.sum()} stimoli)')
cm_f = confusion_matrix(gnn_all_labels[mask_f], gnn_all_preds[mask_f], labels=[0,1])
sns.heatmap(cm_f, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
            xticklabels=class_names, yticklabels=class_names)
axes[2].set_title(f'GNN: Facce Femminili ({mask_f.sum()} stimoli)')
fpr, tpr, _ = roc_curve(gnn_all_labels, gnn_all_probs)
auc_val = auc(fpr, tpr)
axes[3].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {auc_val:.2f}')
axes[3].plot([0,1],[0,1],'navy',linestyle='--')
axes[3].set_title('GNN: ROC Curve')
axes[3].legend()
plt.tight_layout(); plt.show()

acc_m = accuracy_score(gnn_all_labels[mask_m], gnn_all_preds[mask_m]) if mask_m.sum()>0 else 0
acc_f = accuracy_score(gnn_all_labels[mask_f], gnn_all_preds[mask_f]) if mask_f.sum()>0 else 0
print(f"Bias Genere -> Maschi: {acc_m*100:.1f}% | Femmine: {acc_f*100:.1f}%")

---
# SEZIONE B: Ottimizzazione Res1DCNN

In [ ]:
# ============================================================
# SEZIONE B: Nested CV — Res1DCNN
# ============================================================

outer_gkf = GroupKFold(n_splits=N_FOLDS)

cnn_outer_accs = []
cnn_fold_aucs = []
cnn_best_params_per_fold = []
cnn_all_preds, cnn_all_labels, cnn_all_probs, cnn_all_genders = [], [], [], []

print("=" * 60)
print("SEZIONE B: Nested CV CNN 1D")
print("=" * 60)

t0 = time.time()

for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_gkf.split(X_norm, Y, groups=subjects)):

    print(f"\n[Outer Fold {outer_fold+1}/{N_FOLDS}]")

    def objective_cnn(trial):
        filters1     = trial.suggest_categorical('filters1', [16, 32, 64])
        filters2     = trial.suggest_categorical('filters2', [32, 64, 128])
        kernel_size  = trial.suggest_categorical('kernel_size', [3, 5, 7, 11])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.8)
        lr           = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        epochs       = trial.suggest_categorical('epochs', [60, 80, 100])

        gss_inner = GroupShuffleSplit(n_splits=3, test_size=0.25,
                                      random_state=SEED + outer_fold)
        
        split_val_losses = []

        for local_tr, local_val in gss_inner.split(
                X_norm[outer_train_idx], Y[outer_train_idx], groups=subjects[outer_train_idx]):
            
            inner_train_idx = outer_train_idx[local_tr]
            inner_val_idx   = outer_train_idx[local_val]

            train_ds = EEGDataset(X_norm[inner_train_idx], Y[inner_train_idx], augment=True)
            val_ds   = EEGDataset(X_norm[inner_val_idx],   Y[inner_val_idx],   augment=False)
            train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
            val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

            model = Res1DCNN(filters1=filters1, filters2=filters2,
                             kernel_size=kernel_size, dropout_rate=dropout_rate).to(device)
            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = nn.CrossEntropyLoss()

            for epoch in range(epochs):
                model.train()
                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()
                    criterion(model(inputs), labels).backward()
                    optimizer.step()
                    
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * inputs.size(0)
                    
            val_loss /= len(val_loader.dataset)
            split_val_losses.append(val_loss)
            
        return np.mean(split_val_losses)

    # DIRECTION = MINIMIZE
    study = optuna.create_study(direction='minimize',
                                study_name=f'CNN_outer{outer_fold}')
    study.optimize(objective_cnn, n_trials=N_INNER_TRIALS, n_jobs=1)

    best_p = study.best_trial.params
    cnn_best_params_per_fold.append(best_p)
    print(f"  Inner best val loss: {study.best_trial.value:.4f}  |  HP: {best_p}")

    torch.manual_seed(SEED + outer_fold)
    model_final = Res1DCNN(
        filters1=best_p['filters1'], filters2=best_p['filters2'],
        kernel_size=best_p['kernel_size'], dropout_rate=best_p['dropout_rate']
    ).to(device)

    outer_train_ds = EEGDataset(X_norm[outer_train_idx], Y[outer_train_idx], augment=True)
    outer_test_ds  = EEGDataset(X_norm[outer_test_idx],  Y[outer_test_idx],  augment=False)
    outer_train_loader = DataLoader(outer_train_ds, batch_size=16, shuffle=True)
    outer_test_loader  = DataLoader(outer_test_ds,  batch_size=16, shuffle=False)

    optimizer = optim.Adam(model_final.parameters(),
                           lr=best_p['lr'], weight_decay=best_p['weight_decay'])
    criterion = nn.CrossEntropyLoss()

    for epoch in range(best_p['epochs']):
        model_final.train()
        for inputs, labels in outer_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            criterion(model_final(inputs), labels).backward()
            optimizer.step()
            
    model_final.eval()
    c = 0; t = 0
    fold_probs, fold_labels, fold_preds = [], [], []
    
    with torch.no_grad():
        for inputs, labels in outer_test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            out = model_final(inputs)
            preds = out.argmax(1)
            
            c += (preds == labels).sum().item()
            t += labels.size(0)
            
            fold_probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            fold_preds.extend(preds.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())
            
    best_outer_acc = c / t
    try:
        fold_auc = roc_auc_score(fold_labels, fold_probs)
    except ValueError:
        fold_auc = 0.5
        
    cnn_outer_accs.append(best_outer_acc)
    cnn_fold_aucs.append(fold_auc)
    
    cnn_all_probs.extend(fold_probs)
    cnn_all_preds.extend(fold_preds)
    cnn_all_labels.extend(fold_labels)
    cnn_all_genders.extend(Y_gender[outer_test_idx])
    
    print(f"  Outer test acc: {best_outer_acc*100:.1f}% (AUC: {fold_auc:.3f})")

In [ ]:
cnn_all_preds   = np.array(cnn_all_preds)
cnn_all_labels  = np.array(cnn_all_labels)
cnn_all_probs   = np.array(cnn_all_probs)
cnn_all_genders = np.array(cnn_all_genders)

elapsed_cnn = time.time() - t0
print(f"\nCNN Nested CV Accuracy: "
      f"{np.mean(cnn_outer_accs)*100:.1f}% ± {np.std(cnn_outer_accs)*100:.1f}%")
print(f"Fold Details: {[f'{a*100:.0f}%' for a in cnn_outer_accs]}")
print(f"Tempo totale: {elapsed_cnn/60:.1f} min")
print()
print(classification_report(cnn_all_labels, cnn_all_preds, target_names=class_names))

# Plot (identico a prima)
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
cm = confusion_matrix(cnn_all_labels, cnn_all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('CNN: Confusion Matrix (Nested CV)')
mask_m = cnn_all_genders == 'Male'
mask_f = cnn_all_genders == 'Female'
cm_m = confusion_matrix(cnn_all_labels[mask_m], cnn_all_preds[mask_m], labels=[0,1])
sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title(f'CNN: Facce Maschili ({mask_m.sum()} stimoli)')
cm_f = confusion_matrix(cnn_all_labels[mask_f], cnn_all_preds[mask_f], labels=[0,1])
sns.heatmap(cm_f, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
            xticklabels=class_names, yticklabels=class_names)
axes[2].set_title(f'CNN: Facce Femminili ({mask_f.sum()} stimoli)')
fpr, tpr, _ = roc_curve(cnn_all_labels, cnn_all_probs)
auc_val = auc(fpr, tpr)
axes[3].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {auc_val:.2f}')
axes[3].plot([0,1],[0,1],'navy',linestyle='--')
axes[3].set_title('CNN: ROC Curve')
axes[3].legend()
plt.tight_layout(); plt.show()

acc_m = accuracy_score(cnn_all_labels[mask_m], cnn_all_preds[mask_m]) if mask_m.sum()>0 else 0
acc_f = accuracy_score(cnn_all_labels[mask_f], cnn_all_preds[mask_f]) if mask_f.sum()>0 else 0
print(f"Bias Genere -> Maschi: {acc_m*100:.1f}% | Femmine: {acc_f*100:.1f}%")

---
# SEZIONE C: Ottimizzazione MemoryRNN (LSTM)

In [ ]:
# ============================================================
# SEZIONE C: Nested CV — MemoryRNN (LSTM)
# ============================================================
N_INNER_TRIALS = 10  # Mantenuto a 10 trial come richiesto

outer_gkf = GroupKFold(n_splits=N_FOLDS)

rnn_outer_accs = []
rnn_fold_aucs = []
rnn_best_params_per_fold = []
rnn_all_preds, rnn_all_labels, rnn_all_probs, rnn_all_genders = [], [], [], []

print("=" * 60)
print("SEZIONE C: Nested CV RNN (LSTM)")
print("=" * 60)

t0 = time.time()

for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_gkf.split(X_norm, Y, groups=subjects)):

    print(f"\n[Outer Fold {outer_fold+1}/{N_FOLDS}]")

    def objective_rnn(trial):
        hidden_size  = trial.suggest_categorical('hidden_size', [32, 64, 128])
        num_layers   = trial.suggest_categorical('num_layers', [1, 2])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.8)
        lr           = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        epochs       = trial.suggest_categorical('epochs', [60, 80, 100])

        gss_inner = GroupShuffleSplit(n_splits=3, test_size=0.25,
                                      random_state=SEED + outer_fold)
        
        split_val_losses = []

        for local_tr, local_val in gss_inner.split(
                X_norm[outer_train_idx], Y[outer_train_idx], groups=subjects[outer_train_idx]):
            
            inner_train_idx = outer_train_idx[local_tr]
            inner_val_idx   = outer_train_idx[local_val]

            train_ds = EEGDataset(X_norm[inner_train_idx], Y[inner_train_idx], augment=True)
            val_ds   = EEGDataset(X_norm[inner_val_idx],   Y[inner_val_idx],   augment=False)
            train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
            val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

            model = MemoryRNN(hidden_size=hidden_size, num_layers=num_layers,
                              dropout_rate=dropout_rate).to(device)
            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = nn.CrossEntropyLoss()

            for epoch in range(epochs):
                model.train()
                for inputs, labels in train_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()
                    criterion(model(inputs), labels).backward()
                    optimizer.step()
                    
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * inputs.size(0)
                    
            val_loss /= len(val_loader.dataset)
            split_val_losses.append(val_loss)
            
        return np.mean(split_val_losses)

    # DIRECTION = MINIMIZE
    study = optuna.create_study(direction='minimize',
                                study_name=f'RNN_outer{outer_fold}')
    study.optimize(objective_rnn, n_trials=N_INNER_TRIALS, n_jobs=1)

    best_p = study.best_trial.params
    rnn_best_params_per_fold.append(best_p)
    print(f"  Inner best val loss: {study.best_trial.value:.4f}  |  HP: {best_p}")

    torch.manual_seed(SEED + outer_fold)
    model_final = MemoryRNN(
        hidden_size=best_p['hidden_size'], num_layers=best_p['num_layers'],
        dropout_rate=best_p['dropout_rate']
    ).to(device)

    outer_train_ds = EEGDataset(X_norm[outer_train_idx], Y[outer_train_idx], augment=True)
    outer_test_ds  = EEGDataset(X_norm[outer_test_idx],  Y[outer_test_idx],  augment=False)
    outer_train_loader = DataLoader(outer_train_ds, batch_size=16, shuffle=True)
    outer_test_loader  = DataLoader(outer_test_ds,  batch_size=16, shuffle=False)

    optimizer = optim.Adam(model_final.parameters(),
                           lr=best_p['lr'], weight_decay=best_p['weight_decay'])
    criterion = nn.CrossEntropyLoss()

    for epoch in range(best_p['epochs']):
        model_final.train()
        for inputs, labels in outer_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            criterion(model_final(inputs), labels).backward()
            optimizer.step()

    model_final.eval()
    c = 0; t = 0
    fold_probs, fold_labels, fold_preds = [], [], []
    
    with torch.no_grad():
        for inputs, labels in outer_test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            out = model_final(inputs)
            preds = out.argmax(1)
            
            c += (preds == labels).sum().item()
            t += labels.size(0)
            
            fold_probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            fold_preds.extend(preds.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())
            
    best_outer_acc = c / t
    try:
        fold_auc = roc_auc_score(fold_labels, fold_probs)
    except ValueError:
        fold_auc = 0.5
        
    rnn_outer_accs.append(best_outer_acc)
    rnn_fold_aucs.append(fold_auc)
    
    rnn_all_probs.extend(fold_probs)
    rnn_all_preds.extend(fold_preds)
    rnn_all_labels.extend(fold_labels)
    rnn_all_genders.extend(Y_gender[outer_test_idx])
    
    print(f"  Outer test acc: {best_outer_acc*100:.1f}% (AUC: {fold_auc:.3f})")

In [ ]:
rnn_all_preds   = np.array(rnn_all_preds)
rnn_all_labels  = np.array(rnn_all_labels)
rnn_all_probs   = np.array(rnn_all_probs)
rnn_all_genders = np.array(rnn_all_genders)

elapsed_rnn = time.time() - t0
print(f"\nRNN Nested CV Accuracy: "
      f"{np.mean(rnn_outer_accs)*100:.1f}% ± {np.std(rnn_outer_accs)*100:.1f}%")
print(f"Fold Details: {[f'{a*100:.0f}%' for a in rnn_outer_accs]}")
print(f"Tempo totale: {elapsed_rnn/60:.1f} min")
print()
print(classification_report(rnn_all_labels, rnn_all_preds, target_names=class_names))

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
cm = confusion_matrix(rnn_all_labels, rnn_all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('RNN: Confusion Matrix (Nested CV)')
mask_m = rnn_all_genders == 'Male'
mask_f = rnn_all_genders == 'Female'
cm_m = confusion_matrix(rnn_all_labels[mask_m], rnn_all_preds[mask_m], labels=[0,1])
sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title(f'RNN: Facce Maschili ({mask_m.sum()} stimoli)')
cm_f = confusion_matrix(rnn_all_labels[mask_f], rnn_all_preds[mask_f], labels=[0,1])
sns.heatmap(cm_f, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
            xticklabels=class_names, yticklabels=class_names)
axes[2].set_title(f'RNN: Facce Femminili ({mask_f.sum()} stimoli)')
fpr, tpr, _ = roc_curve(rnn_all_labels, rnn_all_probs)
auc_val = auc(fpr, tpr)
axes[3].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {auc_val:.2f}')
axes[3].plot([0,1],[0,1],'navy',linestyle='--')
axes[3].set_title('RNN: ROC Curve')
axes[3].legend()
plt.tight_layout(); plt.show()

acc_m = accuracy_score(rnn_all_labels[mask_m], rnn_all_preds[mask_m]) if mask_m.sum()>0 else 0
acc_f = accuracy_score(rnn_all_labels[mask_f], rnn_all_preds[mask_f]) if mask_f.sum()>0 else 0
print(f"Bias Genere -> Maschi: {acc_m*100:.1f}% | Femmine: {acc_f*100:.1f}%")

---
# Confronto Finale: Le 3 Archietture Dopo Ottimizzazione
Riepilogo comparativo con barre d'errore (deviazione standard sui fold) e overlay delle ROC curve.

In [ ]:
# Riepilogo Grafico
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar Plot con Error Bars
model_names = ['GNN', 'CNN 1D', 'RNN (LSTM)']
means = [np.mean(gnn_outer_accs), np.mean(cnn_outer_accs), np.mean(rnn_outer_accs)]  # ← outer_accs
stds  = [np.std(gnn_outer_accs),  np.std(cnn_outer_accs),  np.std(rnn_outer_accs)]   # ← outer_accs
colors = ['purple', 'salmon', 'lightgreen']

bars = axes[0].bar(model_names, means, yerr=stds, capsize=10, color=colors,
                   edgecolor='black', linewidth=0.8)
for bar, m, s in zip(bars, means, stds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.02,
                 f'{m*100:.1f}±{s*100:.1f}%', ha='center', fontweight='bold')
axes[0].set_ylabel('Accuracy (Nested CV — Outer Fold Mean)')  # ← label aggiornata
axes[0].set_title('Confronto Post-Ottimizzazione (Nested CV)')
axes[0].set_ylim([0, 1.15])
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# ROC Curves Overlay — invariate, usano già gnn_all_probs etc.
fpr_g, tpr_g, _ = roc_curve(gnn_all_labels, gnn_all_probs)
fpr_c, tpr_c, _ = roc_curve(cnn_all_labels, cnn_all_probs)
fpr_r, tpr_r, _ = roc_curve(rnn_all_labels, rnn_all_probs)

auc_gnn_mean = np.mean(gnn_fold_aucs)
auc_cnn_mean = np.mean(cnn_fold_aucs)
auc_rnn_mean = np.mean(rnn_fold_aucs)

axes[1].plot(fpr_g, tpr_g, 'purple', lw=2, label=f'GNN (Mean AUC={auc_gnn_mean:.2f})')
axes[1].plot(fpr_c, tpr_c, 'red',    lw=2, label=f'CNN (Mean AUC={auc_cnn_mean:.2f})')
axes[1].plot(fpr_r, tpr_r, 'green',  lw=2, label=f'RNN (Mean AUC={auc_rnn_mean:.2f})')
axes[1].plot([0, 1], [0, 1], 'navy', linestyle='--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves Comparate')
axes[1].legend()

plt.tight_layout()
plt.show()

# Tabella Gender Bias — invariata
print("\n" + "=" * 60)
print("BIAS DI GENERE (Post-Hoc Analysis)")
print("=" * 60)
print(f"{'Modello':15s} | {'Facce M':10s} | {'Facce F':10s} | {'Delta':10s}")
print("-" * 50)

for name, preds, labels, genders in [
    ('GNN', gnn_all_preds, gnn_all_labels, gnn_all_genders),
    ('CNN', cnn_all_preds, cnn_all_labels, cnn_all_genders),
    ('RNN', rnn_all_preds, rnn_all_labels, rnn_all_genders)]:
    m_mask = genders == 'Male'
    f_mask = genders == 'Female'
    acc_m = accuracy_score(labels[m_mask], preds[m_mask]) if m_mask.sum() > 0 else 0
    acc_f = accuracy_score(labels[f_mask], preds[f_mask]) if f_mask.sum() > 0 else 0
    delta = abs(acc_m - acc_f)
    print(f"{name:15s} | {acc_m*100:8.1f}% | {acc_f*100:8.1f}% | {delta*100:8.1f}%")

try:
    total_time = elapsed_gnn + elapsed_cnn + elapsed_rnn
    print(f"\nTempo totale di ottimizzazione: {total_time/3600:.1f} ore")
except:
    pass

---
# Conclusioni Intermedie (Risultati Classificazione)

## 1. Il Cervello Distingue l'IA? — Sì.

Tutti e 3 i modelli, dopo ottimizzazione bayesiana con validazione GroupKFold su 5 fold (26 soggetti, zero data leakage), superano significativamente il livello di casualità (50%):

| Modello | KFold Accuracy | σ | Best Trial (Optuna) | Tempo |
|---------|---------------|---|---------------------|-------|
| **GNN (TemporalGraphNet)** | **74.0% ± 3.7%** | Bassa ✅ | 78.2% | 178 min |
| RNN (LSTM) | 69.3% ± 6.0% | Alta ⚠️ | 75.3% | 487 min |
| CNN 1D | 66.9% ± 4.6% | Media | 75.2% | 150 min |

La **GNN** domina sia in accuracy assoluta che in stabilità (deviazione standard più bassa). Il fatto che un modello che apprende le *connessioni spaziali tra elettrodi* superi modelli puramente temporali (LSTM) o puramente locali (CNN) suggerisce che la distinzione AI/Umano nel cervello è un fenomeno di **rete neurale distribuita**, non localizzato in un singolo canale o istante.

## 2. Simmetria Classificatoria

La GNN mostra un equilibrio quasi perfetto tra le classi:
- **AI**: Precision 0.72, Recall 0.75
- **Human**: Precision 0.76, Recall 0.73

CNN e RNN invece mostrano un forte **bias verso la classe Human** (Recall 0.77-0.83 per Human, ma solo 0.54-0.56 per AI). Questo indica che CNN e RNN tendono a classificare "Human" nel dubbio, mentre la GNN riesce a identificare attivamente i pattern cerebrali associati alla visione di un volto artificiale.

## 3. Bias di Genere (Post-Hoc)

| Modello | Maschi | Femmine | Δ |
|---------|--------|---------|---|
| GNN | 75.5% | 72.5% | 3.0% |
| CNN | 63.3% | 70.6% | 7.3% |
| RNN | 71.4% | 66.7% | 4.8% |

- La **GNN** è il modello più equo (Δ = 3.0%), con una lieve preferenza per i volti maschili.
- La **CNN** mostra il bias più marcato (7.3%), ma **invertito**: classifica meglio i volti femminili. Ipotesi: le feature convoluzionali locali catturano meglio le differenze morfologiche dei volti femminili (che in letteratura presentano componenti N170 più pronunciate).
- La **RNN** è intermedia (4.8%), con preferenza per i volti maschili.

## 4. Iperparametri Ottimali — Cosa ci dicono?

- **GNN**: `temp_filters=128, kernel_size=8, dropout=0.55, lr≈0.002, weight_decay≈1e-6`
  - Kernel piccolo (8 timestep ≈ 30ms): la GNN cerca micropattern temporali, delegando la visione d'insieme alla matrice di adiacenza spaziale.
  - Weight decay quasi nullo: la regolarizzazione L2 non serve perché l'AdaptiveAvgPool e il dropout sono sufficienti.
  
- **CNN**: `filters1=16, filters2=32, kernel_size=11, dropout=0.53, weight_decay≈0.009`
  - Architettura paradossalmente *piccola* (16→32 filtri). Con 100 campioni, meno parametri = meno overfitting.
  - Kernel largo (11 timestep ≈ 43ms): la CNN ha bisogno di finestre temporali più ampie perché non ha la matrice spaziale della GNN.
  - Weight decay alto (0.009): forte regolarizzazione necessaria senza la struttura a grafo.

- **RNN**: `hidden_size=128, num_layers=2, dropout=0.53, lr≈0.0006`
  - 2 layer LSTM: la rete ha bisogno di profondità ricorrente per catturare le dipendenze temporali complesse.
  - Learning rate basso (6e-4): le LSTM sono notoriamente sensibili a LR alti (esplosione dei gradienti).

## 5. Permutation Importance — Dove guarda il cervello?

Dall'analisi pre-ottimizzazione della GNN (vincitrice):
- **Finestra critica: W3 (320-480ms)** — corrisponde esattamente alla latenza della **P300/N400**, le componenti cognitive di valutazione e riconoscimento semantico.
- **La finestra N170 (160-320ms) confonde il modello**: corromperla *migliora* l'accuracy, suggerendo che i pattern di riconoscimento facciale generico interferiscono con la classificazione AI/Umano.
- **Le componenti tardive (>480ms) non contribuiscono** alla classificazione.

## 6. Limitazioni e Prossimi Passi

- **Dataset ridotto** (100 campioni, 26 soggetti): i risultati sono robusti grazie alla GroupKFold, ma la generalizzabilità richiede validazione esterna.
- **Varianza di ricostruzione**: la GNN ottimizzata raggiunge 78.2% durante Optuna ma 74.0% alla ricostruzione, a causa della stocasticità dell'inizializzazione dei pesi su dataset piccoli.
- **Prossimi passi**: analisi XAI (Saliency Maps, Integrated Gradients) per mappare con precisione canale×tempo dove il modello concentra l'attenzione, e verifica della Permutation Importance post-ottimizzazione.


---
# Fase 2: Interpretabilità e XAI (Focus sul GNN Ottimizzato)

## 2A. Permutation Importance Post-Ottimizzazione
Ricalcoliamo la Permutation Importance sulla GNN con i **migliori iperparametri trovati da Optuna**, confrontando il risultato con il modello pre-ottimizzazione.

In [ ]:
# Nomi degli elettrodi nell'ordine dei canali del tensore (19 canali)
# NOTA: Il canale 19 indica il sesso dello STIMOLO (FaceSEX), non del partecipante.
channel_names = ['O1', 'O2', 'PO9', 'PO10', 'TP7', 'TP8', 'P3', 'P4', 'AF3', 'AF4', 'AFF1h', 'AFF2h', 'AFF3h', 'AFF4h', 'PCA_Frontal', 'PCA_Parietal', 'PCA_Occipital', 'PCA_Temporal', 'FaceSEX']
# Verifica che corrispondano ai 19 canali
assert len(channel_names) == X.shape[1], f"Attesi {X.shape[1]} canali, definiti {len(channel_names)}"
print(f"Canali definiti: {channel_names}")

In [ ]:
median_acc = np.median(gnn_outer_accs)
representative_fold = int(np.argmin(np.abs(np.array(gnn_outer_accs) - median_acc)))
p_gnn = gnn_best_params_per_fold[representative_fold]

print(f"Fold rappresentativo: {representative_fold+1}/{N_FOLDS}")
print(f"  Accuracy outer: {gnn_outer_accs[representative_fold]*100:.1f}%")
print(f"  Mediana outer:  {median_acc*100:.1f}%")
print(f"  HP selezionati: {p_gnn}")
print(f"\nAccuracy tutti i fold: {[f'{a*100:.0f}%' for a in gnn_outer_accs]}")

In [ ]:
# 2A. Permutation Importance — riusa modelli dalla Nested CV
all_drops_per_fold = []
outer_gkf = GroupKFold(n_splits=N_FOLDS)

for fold_idx, (_, test_idx) in enumerate(outer_gkf.split(X_norm, Y, groups=subjects)):
    model_pi = gnn_outer_models[fold_idx]  # nessun retraining
    model_pi.eval()

    test_ds     = EEGDataset(X_norm[test_idx], Y[test_idx], augment=False)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    X_test_t = torch.cat([x for x, _ in test_loader]).to(device)
    Y_test_t = torch.cat([y for _, y in test_loader]).to(device)

    with torch.no_grad():
        acc_base = accuracy_score(Y_test_t.cpu(),
                                  model_pi(X_test_t).argmax(1).cpu())

    fold_drops  = []
    window_size = 41

    for w in range(5):
        s, e  = w * window_size, (w + 1) * window_size
        X_c   = X_test_t.clone()
        X_c[:, :, s:e] = torch.randn_like(X_c[:, :, s:e]) * X_test_t.std()
        with torch.no_grad():
            acc_c = accuracy_score(Y_test_t.cpu(),
                                   model_pi(X_c).argmax(1).cpu())
        fold_drops.append(acc_base - acc_c)

    all_drops_per_fold.append(fold_drops)

# Plot — invariato
mean_drops = np.mean(all_drops_per_fold, axis=0)
std_drops  = np.std(all_drops_per_fold,  axis=0)

wlabels = [f"W{w+1}\n({int(200+(w*41)*(400/205))}-{int(200+((w+1)*41)*(400/205))} ms)"
           for w in range(5)]

plt.figure(figsize=(8, 5))
plt.errorbar(wlabels, mean_drops, yerr=std_drops,
             fmt='ro-', linewidth=2, markersize=8, capsize=6)
plt.fill_between(wlabels, mean_drops-std_drops, mean_drops+std_drops,
                 color='red', alpha=0.1)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.title('Permutation Importance — GNN (Nested CV, Media ± SD sui Fold)')
plt.ylabel('Drop Accuracy')
plt.xlabel('Finestre Temporali (ms da onset stimolo)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

for i, (m, s) in enumerate(zip(mean_drops, std_drops)):
    ms_start = int(200 + (i*41)*(400/205))
    ms_end   = int(200 + ((i+1)*41)*(400/205))
    print(f"  W{i+1} ({ms_start}-{ms_end} ms): Drop = {m*100:+.1f}% ± {s*100:.1f}%")

## 2B. Analisi per Canale (Channel Importance)
Per ogni elettrodo, corrompiamo SOLO quel canale su tutta la serie temporale e misuriamo il calo. Questo rivela quali aree cerebrali sono critiche per la classificazione.

In [ ]:
# 2B. Channel Importance — riusa modelli dalla Nested CV
all_ch_drops = []
outer_gkf    = GroupKFold(n_splits=N_FOLDS)

for fold_idx, (_, test_idx) in enumerate(outer_gkf.split(X_norm, Y, groups=subjects)):
    model_ch = gnn_outer_models[fold_idx]
    model_ch.eval()

    test_ds     = EEGDataset(X_norm[test_idx], Y[test_idx], augment=False)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    X_test_t = torch.cat([x for x, _ in test_loader]).to(device)
    Y_test_t = torch.cat([y for _, y in test_loader]).to(device)

    with torch.no_grad():
        acc_base = accuracy_score(Y_test_t.cpu(),
                                  model_ch(X_test_t).argmax(1).cpu())

    ch_drops = []
    for ch in range(19):
        X_c = X_test_t.clone()
        X_c[:, ch, :] = (torch.randn(X_c[:, ch, :].shape).to(device)
                         * X_test_t[:, ch, :].std())
        with torch.no_grad():
            acc_c = accuracy_score(Y_test_t.cpu(),
                                   model_ch(X_c).argmax(1).cpu())
        ch_drops.append(acc_base - acc_c)
    all_ch_drops.append(ch_drops)

# Plot — invariato
mean_ch = np.mean(all_ch_drops, axis=0)
std_ch  = np.std(all_ch_drops,  axis=0)

sorted_idx   = np.argsort(mean_ch)[::-1]
sorted_names = [channel_names[i] for i in sorted_idx]
sorted_means = mean_ch[sorted_idx]
sorted_stds  = std_ch[sorted_idx]

plt.figure(figsize=(12, 5))
colors = ['#d32f2f' if m > 0 else '#1976d2' for m in sorted_means]
plt.barh(range(19), sorted_means, xerr=sorted_stds, capsize=4,
         color=colors, edgecolor='black', linewidth=0.5)
plt.yticks(range(19), sorted_names)
plt.xlabel('Drop Accuracy (Importanza)')
plt.title('Channel Importance — GNN (Nested CV)')
plt.axvline(x=0, color='gray', linestyle='--')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 5 canali più importanti:")
for i in range(5):
    print(f"  {sorted_names[i]:8s}: {sorted_means[i]*100:+.2f}% ± {sorted_stds[i]*100:.2f}%")

## 2C. Heatmap Canale × Tempo (Mappa Spaziotemporale)
La mappa definitiva: per ogni combinazione (elettrodo, finestra temporale) misuriamo quanto il modello dipende da quella specifica porzione del segnale EEG.

In [ ]:
# 2C. Heatmap Canale × Tempo — riusa modelli dalla Nested CV
n_time_bins  = 10
bin_size     = 205 // n_time_bins
all_heatmaps = []
outer_gkf    = GroupKFold(n_splits=N_FOLDS)

for fold_idx, (_, test_idx) in enumerate(outer_gkf.split(X_norm, Y, groups=subjects)):
    model_ht = gnn_outer_models[fold_idx]
    model_ht.eval()

    test_ds     = EEGDataset(X_norm[test_idx], Y[test_idx], augment=False)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    X_test_t = torch.cat([x for x, _ in test_loader]).to(device)
    Y_test_t = torch.cat([y for _, y in test_loader]).to(device)

    with torch.no_grad():
        acc_base = accuracy_score(Y_test_t.cpu(),
                                  model_ht(X_test_t).argmax(1).cpu())

    heatmap = np.zeros((19, n_time_bins))
    for ch in range(19):
        for tb in range(n_time_bins):
            s   = tb * bin_size
            e   = min(s + bin_size, 205)
            X_c = X_test_t.clone()
            X_c[:, ch, s:e] = (torch.randn_like(X_c[:, ch, s:e])
                                * X_test_t[:, ch, s:e].std())
            with torch.no_grad():
                acc_c = accuracy_score(Y_test_t.cpu(),
                                       model_ht(X_c).argmax(1).cpu())
            heatmap[ch, tb] = acc_base - acc_c
    all_heatmaps.append(heatmap)

# Plot — invariato
mean_heatmap = np.mean(all_heatmaps, axis=0)
ms_per_step  = 400 / 205
time_labels  = [
    f"{int(200+i*bin_size*ms_per_step)}-{int(200+min((i+1)*bin_size,205)*ms_per_step)}ms"
    for i in range(n_time_bins)
]

plt.figure(figsize=(14, 8))
sns.heatmap(mean_heatmap, xticklabels=time_labels, yticklabels=channel_names,
            cmap='RdBu_r', center=0, annot=True, fmt='.2f', linewidths=0.5,
            cbar_kws={'label': 'Drop Accuracy'})
plt.title('Mappa Spaziotemporale — GNN (Nested CV, Canale × Tempo)')
plt.xlabel('Finestra Temporale')
plt.ylabel('Canale EEG')
plt.tight_layout()
plt.show()

flat_idx = np.argsort(mean_heatmap.flatten())[::-1][:5]
print("\nTop 5 zone critiche (canale, tempo):")
for idx in flat_idx:
    ch, tb = divmod(idx, n_time_bins)
    print(f"  {channel_names[ch]:8s} @ {time_labels[tb]:15s}: "
          f"Drop = {mean_heatmap[ch, tb]*100:+.2f}%")

## 2D. XAI — Explainable AI

### Saliency Maps (Input Gradient)
Calcoliamo il gradiente della Loss rispetto all'input: $\frac{\partial \mathcal{L}}{\partial X}$. I valori assoluti del gradiente indicano dove piccole variazioni nell'input causano grandi cambiamenti nella decisione del modello.

In [ ]:
# =========================
# 2D. Saliency Maps
# modello dal fold rappresentativo (già addestrato, nessun retraining)
# =========================
from matplotlib.colors import TwoSlopeNorm

# Recupera test_idx del fold rappresentativo
outer_gkf        = GroupKFold(n_splits=N_FOLDS)
all_outer_splits = list(outer_gkf.split(X_norm, Y, groups=subjects))
_, xai_test_idx  = all_outer_splits[representative_fold]

model_xai = gnn_outer_models[representative_fold]
model_xai.eval()

X_test_all = torch.tensor(X_norm[xai_test_idx], dtype=torch.float32).to(device)
Y_test_all = Y[xai_test_idx]

print(f"Saliency Maps — Fold {representative_fold+1} "
      f"({len(xai_test_idx)} campioni, "
      f"acc={gnn_outer_accs[representative_fold]*100:.1f}%)")

# =========================
# Calcolo gradienti (target = true_class, coerente con IG)
# =========================
class_map     = {0: 'AI', 1: 'Human'}
saliency_maps = {'AI': [], 'Human': []}

for i in range(len(X_test_all)):
    x_sample   = X_test_all[i:i+1].clone().detach().requires_grad_(True)
    true_class = int(Y_test_all[i])

    output = model_xai(x_sample)
    model_xai.zero_grad()
    output[0, true_class].backward()          # ← true_class, non pred_class

    grad       = x_sample.grad.detach().abs().cpu().numpy()[0]  # [19, 205]
    label_name = class_map[true_class]
    saliency_maps[label_name].append(grad)

sal_ai    = np.mean(saliency_maps['AI'],    axis=0) if saliency_maps['AI']    else np.zeros((19, 205))
sal_human = np.mean(saliency_maps['Human'], axis=0) if saliency_maps['Human'] else np.zeros((19, 205))

# =========================
# Plot
# =========================
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

im0 = axes[0].imshow(sal_ai, aspect='auto', cmap='hot',
                     extent=[200, 600, 18.5, -0.5], interpolation='bilinear')
axes[0].set_yticks(range(19))
axes[0].set_yticklabels(channel_names, fontsize=8)
axes[0].set_xlabel('Tempo (ms)')
axes[0].set_title('Saliency Map — Classe AI')
plt.colorbar(im0, ax=axes[0], label='|Gradiente|')

im1 = axes[1].imshow(sal_human, aspect='auto', cmap='hot',
                     extent=[200, 600, 18.5, -0.5], interpolation='bilinear')
axes[1].set_yticks(range(19))
axes[1].set_yticklabels(channel_names, fontsize=8)
axes[1].set_xlabel('Tempo (ms)')
axes[1].set_title('Saliency Map — Classe Human')
plt.colorbar(im1, ax=axes[1], label='|Gradiente|')

sal_diff = sal_ai - sal_human
norm = TwoSlopeNorm(vmin=sal_diff.min(), vcenter=0, vmax=sal_diff.max())
im2 = axes[2].imshow(sal_diff, aspect='auto', cmap='RdBu_r', norm=norm,
                     extent=[200, 600, 18.5, -0.5], interpolation='bilinear')
axes[2].set_yticks(range(19))
axes[2].set_yticklabels(channel_names, fontsize=8)
axes[2].set_xlabel('Tempo (ms)')
axes[2].set_title('Differenza Saliency (AI − Human)')
plt.colorbar(im2, ax=axes[2], label='Δ |Gradiente|')

plt.suptitle(f'Saliency Maps — GNN Nested CV (Fold {representative_fold+1})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Integrated Gradients (Sundararajan et al. 2017)
Metodo più rigoroso delle Saliency Maps: integra i gradienti lungo un percorso dall'input *baseline* (zero) all'input reale. Questo elimina artefatti di saturazione e produce attribuzioni che sommano esattamente alla differenza di output.

In [ ]:
def integrated_gradients(model, x_input, target_class, n_steps=50):
    """Calcola Integrated Gradients per un singolo campione."""
    baseline = torch.zeros_like(x_input)  # Baseline = zero signal
    
    # Interpola tra baseline e input
    scaled_inputs = [baseline + (float(i) / n_steps) * (x_input - baseline) 
                     for i in range(n_steps + 1)]
    
    grads = []
    for scaled in scaled_inputs:
        scaled = scaled.clone().requires_grad_(True)
        output = model(scaled)
        model.zero_grad()
        output[0, target_class].backward()
        grads.append(scaled.grad.data.clone())
    
    # Integrazione trapezoidale
    grads = torch.stack(grads)
    avg_grads = (grads[:-1] + grads[1:]) / 2.0
    integrated = avg_grads.mean(dim=0)
    
    # Attribuzione = (input - baseline) * gradiente integrato
    attribution = (x_input - baseline) * integrated
    return attribution.cpu().numpy()[0]

# Calcolo IG per ogni campione del test set
model_xai.eval()
ig_maps = {'AI': [], 'Human': []}

print("Calcolo Integrated Gradients (potrebbe richiedere qualche minuto)...")
for i in range(len(X_test_all)):
    x_sample = X_test_all[i:i+1].clone()
    true_class = Y_test_all[i]
    
    ig = integrated_gradients(model_xai, x_sample, target_class=true_class, n_steps=50)
    
    label_name = 'AI' if true_class == 0 else 'Human'
    ig_maps[label_name].append(np.abs(ig))
    
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(X_test_all)} campioni elaborati")

ig_ai = np.mean(ig_maps['AI'], axis=0) if ig_maps['AI'] else np.zeros((19, 205))
ig_human = np.mean(ig_maps['Human'], axis=0) if ig_maps['Human'] else np.zeros((19, 205))

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

im0 = axes[0].imshow(ig_ai, aspect='auto', cmap='hot',
                      extent=[200, 600, 18.5, -0.5], interpolation='bilinear')
axes[0].set_yticks(range(19))
axes[0].set_yticklabels(channel_names, fontsize=8)
axes[0].set_xlabel('Tempo (ms)')
axes[0].set_title('Integrated Gradients — Classe AI')
plt.colorbar(im0, ax=axes[0], label='Attribuzione')

im1 = axes[1].imshow(ig_human, aspect='auto', cmap='hot',
                      extent=[200, 600, 18.5, -0.5], interpolation='bilinear')
axes[1].set_yticks(range(19))
axes[1].set_yticklabels(channel_names, fontsize=8)
axes[1].set_xlabel('Tempo (ms)')
axes[1].set_title('Integrated Gradients — Classe Human')
plt.colorbar(im1, ax=axes[1], label='Attribuzione')

ig_diff = ig_ai - ig_human
vmax = max(abs(ig_diff.min()), abs(ig_diff.max()))
im2 = axes[2].imshow(ig_diff, aspect='auto', cmap='RdBu_r',
                      extent=[200, 600, 18.5, -0.5], interpolation='bilinear',
                      vmin=-vmax, vmax=vmax)
axes[2].set_yticks(range(19))
axes[2].set_yticklabels(channel_names, fontsize=8)
axes[2].set_xlabel('Tempo (ms)')
axes[2].set_title('Differenza IG (AI − Human)')
plt.colorbar(im2, ax=axes[2], label='Δ Attribuzione')

plt.suptitle(f'Integrated Gradients — GNN Nested CV (Fold {representative_fold+1})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Aggregazione temporale delle attribuzioni
ig_combined = ig_ai + ig_human

# Media per canale (importanza complessiva del canale)
channel_importance_ig = ig_combined.mean(axis=1)
sorted_ig = np.argsort(channel_importance_ig)[::-1]

print("\nClassifica Canali per Integrated Gradients:")
for rank, idx in enumerate(sorted_ig[:10]):
    print(f"  {rank+1}. {channel_names[idx]:8s}: {channel_importance_ig[idx]:.6f}")

# Media nel tempo (importanza per finestra temporale)
time_importance_ig = ig_combined.mean(axis=0)

# Creazione asse X reale in ms
ms_axis = np.linspace(200, 600, 205)

plt.figure(figsize=(12, 4))
plt.plot(ms_axis, time_importance_ig, color='darkorange', linewidth=2)
plt.fill_between(ms_axis, time_importance_ig, alpha=0.2, color='orange')
plt.xlabel('Tempo (ms da onset stimolo)')
plt.ylabel('Attribuzione Media (IG)')
plt.title('Profilo Temporale delle Attribuzioni — GNN')
plt.grid(alpha=0.3)
plt.axvline(x=300, color='red', linestyle='--', alpha=0.5, label='Inizio P300')
plt.axvline(x=450, color='blue', linestyle='--', alpha=0.5, label='Late Positivity (LP)')
plt.legend()
plt.tight_layout()
plt.show()


# Conclusioni Finali: XAI e Interpretazione Neurofisiologica

Alla luce dei risultati effettivi della **Fase 2 (Explainable AI - XAI)** sulla **TemporalGraphNet (GNN) ottimizzata**, analizziamo le evidenze empiriche restituite da ciascuno strumento interpretativo.

---

## 1. Dinamica Temporale: La Finestra Decisiva (Permutation Importance)

Il grafico della **Permutation Importance** su K fold (Immagine 1) rivela un pattern temporale netto e asimmetrico:

- **W1 (200–280 ms)**: Drop ≈ 0.00 ± 0.055. Nessun contributo stabile: la finestra della risposta visiva precoce (P1/N170) è irrilevante per la classificazione, con alta varianza interfold che suggerisce instabilità idiosincratica.
- **W2 (280–360 ms)**: Drop medio ≈ −0.04. La corruzione di questa finestra *migliora* leggermente la performance. I processi di riconoscimento facciale strutturale precoce (N250) sono rumore per il classificatore: il modello distingue meglio l'IA dal reale *ignorando* la risposta percettiva automatica al volto.
- **W3 (360–440 ms)**: Drop ≈ 0.00. Finestra di transizione, nessun contributo apprezzabile.
- **W4 (440–520 ms)**: Drop ≈ −0.02. Ancora leggermente negativo, come per W2: anche la porzione iniziale della componente P300 classica non è l'elemento discriminante.
- **W5 (520–600 ms)**: Drop ≈ **+0.11**, il valore più elevato con ampia distanza sugli altri. Questa è la finestra **critica e necessaria**: corrompere il segnale tra 520 e 600 ms causa un crollo netto della performance. Corrisponde alla **Late Positive Component (LPC/LPP)**, la componente associata alla valutazione cognitiva consapevole, alla categorizzazione semantica e al riconoscimento mnestico. Il cervello "decide" che un volto è artificiale in questa fase tardiva, non durante la percezione automatica.

**Implicazione neurofisiologica**: la discriminazione AI/Umano non è un processo bottom-up di riconoscimento sensoriale, ma un giudizio top-down che emerge solo dopo 500 ms dallo stimolo, nella fase di elaborazione cosciente e valutativa.

---

## 2. Anatomia della Classificazione (Channel Importance)

La **Channel Importance** (Immagine 2) distingue nettamente due gruppi di canali:

**Canali informativi (drop positivo):**
- **P3** è il canale più importante in assoluto (drop > 0.04), seguito da **AF4**, **TP8** e **P4**. L'asse parieto-frontale destro e sinistro — P3/P4 per la valutazione spaziale e semantica, TP8 per l'integrazione temporo-parietale — costituisce la rete neurale che supporta il giudizio di autenticità del volto. Canali supplementari **AFF1h**, **AFF4h**, **AFF2h** e **PCA_Temporal** contribuiscono positivamente in misura minore.

**Canali confusori (drop negativo — la loro corruzione aiuta):**
- **O2** e **PO9** mostrano i drop negativi più profondi (rispettivamente −0.15 e −0.10 circa), seguiti da **PCA_Occipital**, **FaceSEX**, **O1**, **TP7**, **PO10**, **PCA_Frontal** e **PCA_Parietal**. I canali occipitali (O1, O2, PO9, PO10) catturano il processamento visivo primario: queste risposte sensoriali di basso livello — forma, contrasto, struttura retinica del viso — distraggono il classificatore. Il modello performa meglio quando questi segnali sono assenti.
- **FaceSEX** (canale 19, il genere dello *stimolo*, non del soggetto) ha drop negativo: l'informazione sul genere della faccia presentata è un confondente, non un facilitatore. La GNN classifica meglio quando non ne tiene conto.
- **PCA_Parietal** e **PCA_Frontal**, nonostante il nome, risultano anch'esse confusorie, probabilmente perché catturano varianza condivisa tra le due classi non utile alla discriminazione.

---

## 3. Isole Spaziotemporali (Heatmap Canale × Tempo)

La **Mappa Spaziotemporale** (Immagine 3) localizza con precisione le combinazioni canale–finestra che portano informazione reale:

**Zone a drop positivo (informazione critica):**
- **PO9, TP8, PCA_Occipital, O2 @ 551–590 ms** (drop +0.03 ciascuno): la finestra più tarda dell'epoca è quella in cui tutti questi canali convergono nel contribuire alla classificazione. Paradossalmente, canali occipitali che risultavano confusori nella Channel Importance globale diventano informativi *in questa specifica finestra tardiva*, suggerendo che le risposte visive tardive (LPP occipito-parietale) siano qualitativamente diverse da quelle precoci.
- **P4 @ 278–317 ms** e **TP7 @ 278–317 ms** (drop +0.02): un'isola precoce di discriminazione parieto-temporale destra, nella transizione N250→P300.
- **AF4 @ 473–512 ms** e **P3 @ 473–512 ms** (drop +0.02): attivazione frontoparietale nella finestra centrale della P300.

**Zone a drop fortemente negativo (rumore dannoso):**
- **AF3 @ 551–590 ms**: drop −0.05, il picco negativo più intenso dell'intera mappa. La corruzione del canale frontale sinistro nell'ultima finestra *migliora* la classificazione del 5%. AF3 in questa fase veicola probabilmente attività esecutiva frontale che interferisce con il segnale discriminante parietale tardivo.
- **PO9 @ 395–434 ms** (drop −0.04) e **O2 @ 434–473 ms** (drop −0.03): le risposte visive di medio-latenza in questi canali occipitali sono confusori temporalmente localizzati.

---

## 4. Profilo Temporale Fine (Integrated Gradients)

Il **profilo temporale delle attribuzioni IG** (Immagine 4) offre la risoluzione più alta sulla dinamica temporale della classificazione:

- **200–295 ms**: attribuzione crescente ma moderata, con un piccolo picco locale (~0.020). La fase precoce contribuisce in modo marginale.
- **295–320 ms (P300 onset)**: **primo picco principale** (~0.037). La GNN attribuisce importanza significativa immediatamente all'inizio della P300. Non è la P300 classica "di risposta", ma la transizione tra processamento strutturale e valutazione semantica.
- **320–415 ms**: attribuzione sostenuta con picco secondario intorno a 390 ms (~0.038). Il modello è "attivo" per oltre 100 ms consecutivi durante la P300.
- **415–455 ms**: breve calo, poi il **picco assoluto** (~0.040) a ~455 ms, appena oltre la soglia della Late Positivity. Questo è il momento in cui l'attribuzione IG è massima: la GNN basa la propria decisione prevalentemente su ciò che accade intorno ai 450–470 ms.
- **455–560 ms**: discesa progressiva con un plateau (~0.025).
- **560–600 ms**: risalita finale (~0.030), coerente con la LPP tardiva già evidenziata dalla Permutation Importance.

Il profilo IG rivela che il classificatore non si affida a un singolo istante, ma a una **finestra estesa di 150+ ms** centrata tra 300 e 470 ms, con il picco decisionale a ~450 ms.

---

## 5. Firma Differenziale AI vs. Human (Saliency Maps)

Le **Saliency Maps** (Immagine 5) mostrano pattern qualitativamente simili per le due classi — entrambe presentano bande verticali calde intorno a 300 ms e 450 ms — ma la **mappa differenziale (AI − Human)** evidenzia distinzioni importanti:

- **TP7 e TP8 in finestre tardive (>480 ms)**: leggera dominanza rossa (AI > Human), indicando che il modello usa maggiormente le risposte temporo-parietali tardive per identificare i volti artificiali rispetto a quelli reali.
- **PCA_Parietal e PCA_Occipital @ 300–430 ms**: dominanza blu (Human > AI), ovvero le componenti PCA estratte nelle zone parietali e occipitali pesano di più nel riconoscimento dei volti umani nella finestra P300 classica.
- **AF3/AF4 @ 480–550 ms**: alternanza cromatica, suggerendo che l'attività frontale in questa fase è strutturalmente diversa tra le due classi ma non linearmente separabile.

---

## Sintesi: Un Processo di Autenticazione Cognitiva Tardiva

I quattro strumenti XAI convergono su un quadro interpretativo coerente e neurobiologicamente fondato:

**Il cervello non "vede" che un volto è artificiale — lo "giudica".**

La classificazione AI/Human non avviene nelle fasi di elaborazione visiva automatica (< 400 ms), che anzi introducono confusione. Il processo decisionale emerge in tre momenti progressivi:

1. **278–320 ms (P4, TP7)**: prima isola di discriminazione spaziale nelle aree parieto-temporali destre — un "segnale debole" precoce di incongruenza strutturale.
2. **390–470 ms (P3, AF4, P300/LPC onset)**: consolidamento della valutazione semantica e cognitiva, con il picco assoluto delle attribuzioni IG a ~450 ms. Questo è il cuore decisionale del classificatore.
3. **520–600 ms (PO9, TP8, LPP)**: chiusura e ratifica del giudizio attraverso la Late Positive Component — l'unica macro-finestra temporale la cui corruzione causa un crollo netto dell'accuracy (+0.11 di drop).

Il canale **FaceSEX** — genere della faccia stimolo — risulta confusore netto: il cervello non usa questa informazione per distinguere AI da reale, e la rete che tenta di farlo performa peggio. I canali occipitali primari (O1, O2, PO9) sono confusori globali ma diventano informativi *solo* nell'ultima finestra, suggerendo una riorganizzazione qualitativa del processamento visivo tardivo rispetto a quello precoce.